# Information Content metrika (Data Enrichment)

Ovaj notebook demonstrira proces "obogaćivanja" (*enrichment*) baze znanja dodavanjem izračunatih težina čvorovima simptoma. Svakom simptomu u grafu pridružuje se težina (Information Content - IC) izračunata na osnovu njegove učestalosti pojavljivanja u celokupnom korpusu bolesti:

$$IC = \log\left(\frac{Total\_Diseases}{Frequency + 1}\right)$$

Zašto koristimo ovu metriku?

- **Balansiranje specifičnosti**: Sprečava da generički simptomi (npr. temperatura, umor) potisnu ređe, klinički značajnije simptome (npr. specifični osip),

- **Kažnjavanje nespecifičnosti**: Automatski "kažnjava" (dodaje manju težinu) simptome koji se pojavljuju u stotinama različitih bolesti, jer oni imaju malu dijagnostičku vrednost,

- **Proces razlikovanja**: Omogućava sistemu da logički razlikuje specifične kliničke znake od opštih simptoma, što je temelj za preciznu dijagnozu.

Primer:
- Simptom "fever" (Frequency: 30) će imati nizak IC.
- Simptom "dysphonia" (Frequency: 2) će imati visok IC.

## Instalacija paketa

In [ ]:
%pip install neo4j python-dotenv

# Konekcija sa graf bazom

In [4]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()

neo4j_url = os.getenv("NEO4J_URL")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_username, neo4j_password))

## Ukupan broj bolesti (Total Corpus)

In [8]:
total_diseases = 0

with driver.session() as session:
    result = session.run("""
     MATCH (d:Disease) 
     RETURN count(d) as total_diseases
    """)
    total_diseases = result.single()["total_diseases"]

In [3]:
total_diseases

14460

## Računanje i upisivanje težine (IDF) za svaki simptom

In [9]:
with driver.session() as session:
    has_symptom_uri = "http://purl.obolibrary.org/obo/RO_0002452"

    result = session.run("""
        MATCH (s:Symptom)<-[r:n4sch__SCO_RESTRICTION]-(d:Disease)
        WHERE r.onPropertyURI = $has_symptom_uri
        
        WITH s, count(d) AS disease_frequency
        
        WITH s, log(toFloat($total_diseases) / (disease_frequency + 1)) AS ic_score
        
        SET s.weight = ic_score
        
        RETURN count(s) as symptoms_updated
    """, has_symptom_uri=has_symptom_uri, total_diseases=total_diseases)

    updated_count = result.single()["symptoms_updated"]
    print(f"Uspešno izračunato i upisano težina za {updated_count} simptoma.")

Uspešno izračunato i upisano težina za 305 simptoma.


In [10]:
with driver.session() as session:
    result = session.run("""
        MATCH (s:Symptom)
        WHERE s.weight IS NOT NULL
        RETURN s.n4sch__label[0] AS naziv, s.weight AS tezina
        ORDER BY s.weight DESC
    """)

    for record in result:
        print(f"   - {record['naziv']}: {record['tezina']:.4f}")

   - hypergammaglobulinemia: 8.8860
   - papular rash: 8.8860
   - dysphonia: 8.8860
   - bronchopulmonary bleeding: 8.8860
   - alteration of consciousness: 8.8860
   - decreased appetite: 8.8860
   - bloody stool: 8.8860
   - low blood pressure: 8.8860
   - melena: 8.8860
   - memory impairment: 8.8860
   - descending muscle paralysis: 8.8860
   - urinary incontinence: 8.8860
   - hydrocephalus: 8.8860
   - facial paralysis: 8.8860
   - anal abscess: 8.8860
   - nystagmus: 8.8860
   - bone conduction hyperacusis: 8.8860
   - feeding problem: 8.8860
   - hyperesthesia: 8.8860
   - agitation: 8.8860
   - acquired color vision deficiency: 8.8860
   - lesions in mouth: 8.8860
   - diaphoresis: 8.8860
   - sneezing: 8.8860
   - decreased metabolism: 8.8860
   - epididymitis: 8.8860
   - lesions in lung: 8.8860
   - hepatic abscess: 8.8860
   - weight gain: 8.8860
   - cachexia: 8.8860
   - loss of height: 8.8860
   - hypothermia: 8.8860
   - hypersomnia: 8.8860
   - dry mouth: 8.8860
   -